## 1. Hugging Face Login

#### This imports the token from Kaggle Secrets, so make sure the secret is set correctly before running the code.

In [1]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 2. Check GPU

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## 4. Load Tokenizer

In [ ]:
from collections import defaultdict, Counter
import heapq

SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class BytelevelBPE:

    def __init__(self, vocab_size=5000):
        self.vocab_size = vocab_size
        self.specials = SPECIALS.copy()
        self.byte_vocab = [str(i) for i in range(256)]
        self.merges = []
        self.merge_ranks = {}
        self.vocab = []
        self.stoi = {}
        self.itos = {}
        # Maps each BPE symbol to the bytes represented by it
        self.symbol_to_bytes = {}
    
    # 1. WORD -> UTF-8 BYTE SYMBOLS

    def word_to_symbols(self, word):
        byte_seq = word.encode("utf-8")
        return [str(b) for b in byte_seq] + ["</w>"]

    # 2. BUILD WORD FREQUENCIES

    def build_word_freqs(self, corpus):
        word_counter = Counter()
        print("Building word frequencies...")
        for text in corpus:
            if not isinstance(text, str):
                continue

            word_counter.update(text.split())
        print(
            f"Unique whitespace-separated words: "
            f"{len(word_counter):,}"
        )
        # Convert each unique word to byte symbols only once
        word_freqs = {}

        for word, freq in word_counter.items():
            symbols = tuple(self.word_to_symbols(word))
            word_freqs[symbols] = (
                word_freqs.get(symbols, 0) + freq
            )

        return word_freqs


    # 3. PAIRS OCCURRING IN ONE WORD

    @staticmethod
    def get_pairs(symbols):
        return {
            (symbols[i], symbols[i + 1])
            for i in range(len(symbols) - 1)
        }

    # 4. COUNT PAIR OCCURRENCES IN ONE WORD

    @staticmethod
    def pair_counts_in_word(symbols):
        counts = Counter()
        for i in range(len(symbols) - 1):
            counts[
                (symbols[i], symbols[i + 1])
            ] += 1

        return counts

    # 5. MERGE A PAIR INSIDE ONE WORD

    @staticmethod
    def merge_symbols(symbols, pair, merged_symbol):
        new_symbols = []
        i = 0
        while i < len(symbols):
            if (
                i < len(symbols) - 1
                and symbols[i] == pair[0]
                and symbols[i + 1] == pair[1]
            ):
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1

        return tuple(new_symbols)

    # 6. TRAIN OPTIMIZED BPE

    def train(self, corpus):
        word_freqs = self.build_word_freqs(corpus)
        vocab = (
            self.specials
            + self.byte_vocab
            + ["</w>"]
        )
        seen = set(vocab)
        for b in range(256):
            self.symbol_to_bytes[str(b)] = bytes([b])
        self.symbol_to_bytes["</w>"] = b""

        words = []
        frequencies = []
        for symbols, freq in word_freqs.items():
            words.append(symbols)
            frequencies.append(freq)
        del word_freqs

        pair_counts = defaultdict(int)
        pair_to_words = defaultdict(set)

        print("Building initial pair statistics...")

        for word_id, symbols in enumerate(words):
            freq = frequencies[word_id]
            local_counts = self.pair_counts_in_word(symbols)
            for pair, count in local_counts.items():
                pair_counts[pair] += count * freq
                pair_to_words[pair].add(word_id)

        print(
            f"Initial unique pairs: "
            f"{len(pair_counts):,}"
        )

        # MAX HEAP
        #
        # Python has min-heap, so use negative frequency.
        #
        # We keep stale entries and validate them when popped.
        # This avoids rebuilding the heap after every merge.

        heap = []
        for pair, count in pair_counts.items():
            heapq.heappush(heap,(-count, pair))
        target_merges = max(0,self.vocab_size - len(vocab))
        print(f"Target merges: {target_merges:,}")
        while (len(vocab) < self.vocab_size and heap):
            while heap:
                neg_count, pair = heapq.heappop(heap)
                heap_count = -neg_count
                current_count = pair_counts.get(pair,0)
                # Ignore stale heap entries
                if (current_count == heap_count and current_count > 0):
                    break
            else:
                break

            max_freq = pair_counts.get(pair, 0)

            # Same stopping condition as before
            if max_freq <= 1:
                break

            # Create collision-free merged symbol
            merged_symbol = (f"<bpe_{len(self.merges)}>")

            # Byte representation of new symbol
            left_bytes = self.symbol_to_bytes[pair[0]]
            right_bytes = self.symbol_to_bytes[pair[1]]
            self.symbol_to_bytes[merged_symbol] = left_bytes + right_bytes

            self.merges.append(pair)
            self.merge_ranks[pair] = (len(self.merges) - 1,merged_symbol)
            if merged_symbol not in seen:
                seen.add(merged_symbol)
                vocab.append(merged_symbol)

            # ----------------------------------------------------
            # IMPORTANT OPTIMIZATION
            #
            # Only process words containing selected pair.
            # ----------------------------------------------------

            affected_word_ids = list(pair_to_words.get(pair, set()))
            changed_pairs = set()
            for word_id in affected_word_ids:
                old_symbols = words[word_id]
                freq = frequencies[word_id]
                # Pair may have disappeared because of an earlier
                # update involving this word
                if pair not in self.get_pairs(old_symbols):
                    continue
                
                # Statistics BEFORE merge
                old_counts = self.pair_counts_in_word(old_symbols)

                # Merge selected pair
                new_symbols = self.merge_symbols(old_symbols,pair,merged_symbol)
                if new_symbols == old_symbols:
                    continue

                # Statistics AFTER merge
                new_counts = self.pair_counts_in_word(new_symbols)

                # Save updated word
                words[word_id] = new_symbols

                all_pairs = (
                    set(old_counts)
                    | set(new_counts)
                )

                # Update only pairs affected by this word

                for p in all_pairs:

                    old_n = old_counts.get(p, 0)
                    new_n = new_counts.get(p, 0)
                    delta = (
                        new_n - old_n
                    ) * freq

                    if delta != 0:
                        pair_counts[p] += delta
                        changed_pairs.add(p)

                    # Update inverted index
                    if new_n > 0:
                        pair_to_words[p].add(word_id)
                    else:
                        pair_to_words[p].discard(word_id)

            # Selected pair should now be gone from affected words
            pair_to_words[pair].clear()

            # Push updated frequencies into heap
            for p in changed_pairs:
                count = pair_counts.get(p, 0)
                if count > 0:
                    heapq.heappush(heap,(-count, p))

        self.vocab = vocab
        self.vocab_size = len(vocab)
        self.stoi = {
            symbol: i
            for i, symbol in enumerate(vocab)
        }

        self.itos = {
            i: symbol
            for symbol, i in self.stoi.items()
        }

        print("\nTokenizer training complete.")

        print(
            "Final vocabulary size:",
            self.vocab_size
        )

        print(
            "Number of merges:",
            len(self.merges)
        )

    # 7. APPLY BPE TO ONE WORD

    def apply_bpe(self, word):
        symbols = self.word_to_symbols(word)
        while len(symbols) > 1:
            best_pair = None
            best_rank = float("inf")
            best_merged_symbol = None

            for i in range(len(symbols) - 1):
                pair = (symbols[i],symbols[i + 1])
                merge_info = self.merge_ranks.get(pair)
                if merge_info is not None:
                    rank, merged_symbol = merge_info
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = pair
                        best_merged_symbol = merged_symbol

            if best_pair is None:
                break

            symbols = list(
                self.merge_symbols(
                    tuple(symbols),
                    best_pair,
                    best_merged_symbol
                )
            )

        return symbols

    # 8. ENCODE

    def encode(self,text,add_bos=False,add_eos=False):
        ids = []
        if add_bos:
            ids.append(
                self.stoi["<s>"]
            )

        for word in text.split():
            symbols = self.apply_bpe(word)
            for symbol in symbols:
                ids.append(
                    self.stoi.get(
                        symbol,
                        self.stoi["<unk>"]
                    )
                )

        if add_eos:
            ids.append(
                self.stoi["</s>"]
            )

        return ids

    # 9. DECODE

    def decode(self, ids):

        words = []
        current_bytes = bytearray()

        for token_id in ids:
            symbol = self.itos.get(token_id)
            if symbol is None:
                continue
            if symbol in self.specials:
                continue

            # Determine whether this token ends a word
            ends_word = self._symbol_ends_word(symbol)
            byte_value = self.symbol_to_bytes.get(symbol, b"")
            current_bytes.extend(byte_value)

            if ends_word:
                try:
                    word = bytes(current_bytes).decode("utf-8")
                    words.append(word)
                except UnicodeDecodeError:
                    words.append("<unk>")
                current_bytes = bytearray()

        # Remaining bytes
        if current_bytes:
            try:
                words.append(bytes(current_bytes).decode("utf-8"))
            except UnicodeDecodeError:
                words.append("<unk>")

        return " ".join(words)

    # 10. CHECK WHETHER SYMBOL CONTAINS </w> AT END

    def _symbol_ends_word(self, symbol):
        if symbol == "</w>":
            return True
        # Find merge that created this symbol
        for pair, (_, merged) in self.merge_ranks.items():
            if merged == symbol:
                return self._symbol_ends_word(
                    pair[1]
                )

        return False

    # 11. VOCAB SIZE

    def get_vocab_size(self):
        return len(self.vocab)

## 9. Define The Model

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        if self.d_head % 2 != 0:
            raise ValueError("RoPE requires an even head dimension.")

        inv_freq = 1.0 / (
            10000.0 ** (torch.arange(0, self.d_head, 2).float() / self.d_head)
        )
        angles = torch.outer(torch.arange(max_seq_len).float(), inv_freq)
        self.register_buffer("rope_cos", angles.cos(), persistent=False)
        self.register_buffer("rope_sin", angles.sin(), persistent=False)
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

    def apply_rope(self, q, k, offset=0):
        T = q.size(2)
        cos = self.rope_cos[offset:offset + T][None, None].to(q.dtype)
        sin = self.rope_sin[offset:offset + T][None, None].to(q.dtype)

        def rotate(x):
            first, second = x.chunk(2, dim=-1)
            return torch.cat(
                [first * cos - second * sin, first * sin + second * cos],
                dim=-1,
            )

        return rotate(q), rotate(k)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        q, k = self.apply_rope(q, k)

        scores = q @ k.transpose(-2, -1) * self.attention_scale

        scores = scores.masked_fill(self.causal_mask[:T, :T] == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)

        weighted_sum = attn_weights @ v
        weighted_sum = weighted_sum.transpose(1, 2).contiguous().view(B, T, C)

        out = self.out_proj(weighted_sum)
        out = self.resid_dropout(out)

        return out

In [ ]:
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 512
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4          
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "rope"   
    attention_scale: float = None      
    norm_first: bool = False            # norm_first=True is for pre-norm else post-norm


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        # TODO: two linear layers with a GELU in between, plus dropout
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout,
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

        self.norm_first = cfg.norm_first

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))
        else:
            x = self.ln1(x + self.attn(x))
            x = self.ln2(x + self.ffn(x))
        return x


class DecoderLM(nn.Module):
    """The model class the autograder imports directly. Do not rename it."""

    def __init__(self, cfg: LMConfig):
        super().__init__()
        if cfg.pos_emb_type != "rope":
            raise ValueError("This notebook requires a RoPE model/checkpoint.")
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)

        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx: (B, T) LongTensor of token ids -> logits (B, T, vocab_size)."""
        B, T = idx.shape

        x = self.tok_emb(idx)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.head(x)
        return logits


# **Phase 3**

In [6]:
import json
import random
import zipfile
from pathlib import Path
from dataclasses import asdict
from types import MethodType

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import (
    precision_recall_fscore_support,
    classification_report,
)

# Change this to your saved Phase 1 Model 1 folder
MODEL_DIR = Path(
    "/kaggle/input/datasets/bokkatrishalreddy/model1-phase3-saved-files-t"
)

METHOD = "lora"

OUTPUT_DIR = Path(
    f"/kaggle/working/phase3_model1_{METHOD}_target_mean"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
MAX_LENGTH = 512
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 1e-4
PATIENCE = 3

RANK = 8
ALPHA = 16

USE_CONTEXT = True
USE_CLASS_WEIGHTS = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

use_amp = device.type == "cuda"

amp_dtype = (
    torch.bfloat16
    if use_amp and torch.cuda.is_bf16_supported()
    else torch.float16
)

print("Device:", device)

Device: cuda


In [7]:
from huggingface_hub import hf_hub_download

DATA_DIR = Path("/kaggle/working/capstone_data")

files = [
    "Train/CL_train.json",
    "Train/CL_dev.json",
    "Train/IT_train.json",
    "Train/IT_dev.json",
    "Train/label_vocab.json",
    "Test/CL_test.json",
    "Test/IT_test.json",
]

for filename in files:
    hf_hub_download(
        repo_id="Exploration-Lab/CS-781-Capstone",
        repo_type="dataset",
        filename=f"Phase_3_RR/{filename}",
        local_dir=str(DATA_DIR),
        token=globals().get("HF_TOKEN"),
    )

RR_DIR = DATA_DIR / "Phase_3_RR"


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

In [8]:
checkpoint = torch.load(
    MODEL_DIR / "model.pt",
    map_location="cpu",
    weights_only=True,
)

cfg = LMConfig(**checkpoint["model_config"])

phase1_model = DecoderLM(cfg)
phase1_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

del checkpoint

MAX_LENGTH = min(MAX_LENGTH, cfg.max_seq_len)

state = load_json(MODEL_DIR / "tokenizer.json")

tokenizer = BytelevelBPE(state["vocab_size"])
tokenizer.specials = state["specials"]
tokenizer.vocab = state["vocab"]
tokenizer.vocab_size = len(state["vocab"])

tokenizer.stoi = {
    symbol: i for i, symbol in enumerate(tokenizer.vocab)
}

tokenizer.itos = {
    i: symbol for symbol, i in tokenizer.stoi.items()
}

tokenizer.merges = [
    tuple(pair) for pair in state["merges"]
]

tokenizer.merge_ranks = {
    tuple(record["pair"]): (
        record["rank"],
        record["merged_symbol"],
    )
    for record in state["merge_ranks"]
}

tokenizer.symbol_to_bytes = {
    symbol: bytes(values)
    for symbol, values in state["symbol_to_bytes"].items()
}

del state

PAD_ID = tokenizer.stoi["<pad>"]
BOS_ID = tokenizer.stoi["<s>"]
EOS_ID = tokenizer.stoi["</s>"]
UNK_ID = tokenizer.stoi["<unk>"]

if len(tokenizer.vocab) != cfg.vocab_size:
    raise ValueError("Tokenizer and checkpoint vocabulary sizes differ.")

print("Maximum input length:", MAX_LENGTH)

Maximum input length: 512


In [9]:
vocab = load_json(RR_DIR / "Train/label_vocab.json")

if isinstance(vocab, dict):
    for key in ("label2id", "id2label", "labels"):
        if key in vocab and isinstance(vocab[key], (dict, list)):
            vocab = vocab[key]
            break

if isinstance(vocab, list):
    label_names = vocab
elif all(str(key).isdigit() for key in vocab):
    label_names = [
        vocab[key]
        for key in sorted(vocab, key=lambda x: int(x))
    ]
else:
    label_names = sorted(vocab, key=lambda x: int(vocab[x]))

label2id = {
    label: i for i, label in enumerate(label_names)
}
id2label = dict(enumerate(label_names))
NUM_LABELS = len(label_names)


def read_documents(path, domain, labeled=True):
    raw = load_json(path)

    if isinstance(raw, dict) and "id" in raw and "text" in raw:
        records = [raw]
    elif isinstance(raw, dict):
        records = []
        for doc_id, record in raw.items():
            if isinstance(record, list):
                record = {"text": record}
            records.append({**record, "id": doc_id})
    else:
        records = raw

    documents = []
    seen = set()

    for record in records:
        doc_id = str(record["id"])
        sentences = record["text"]
        labels = record["labels"] if labeled else None

        if doc_id in seen:
            raise ValueError(f"Duplicate document ID: {doc_id}")
        seen.add(doc_id)

        if not isinstance(sentences, list):
            raise ValueError("Expected text to be a list of sentences.")

        if labeled:
            if len(sentences) != len(labels):
                raise ValueError(f"Sentence-label mismatch: {doc_id}")
            if any(label not in label2id for label in labels):
                raise ValueError(f"Unknown label in document: {doc_id}")

        documents.append({
            "id": doc_id,
            "domain": domain,
            "text": sentences,
            "labels": labels,
        })

    return documents


train_docs = []
dev_docs = []

for domain in ("CL", "IT"):
    train_docs += read_documents(
        RR_DIR / f"Train/{domain}_train.json", domain
    )
    dev_docs += read_documents(
        RR_DIR / f"Train/{domain}_dev.json", domain
    )

print("Labels:", label_names)
print("Training documents:", len(train_docs))
print("Development documents:", len(dev_docs))

Labels: ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']
Training documents: 80
Development documents: 10


In [10]:
previous_prefix = tokenizer.encode("Previous sentence:")
next_prefix = tokenizer.encode("Next sentence:")
target_prefix = tokenizer.encode("Target sentence:")

# BOS + three EOS separators + textual prefixes
overhead = (
    4
    + len(previous_prefix)
    + len(next_prefix)
    + len(target_prefix)
)

target_capacity = MAX_LENGTH - overhead

if target_capacity < 1:
    raise ValueError("MAX_LENGTH is too small for the input format.")


def make_chunks(target, left, right):
    chunks = []

    for start in range(0, len(target), target_capacity):
        part = target[start:start + target_capacity]
        available = target_capacity - len(part)

        n_left = min(len(left), available // 2)
        n_right = min(len(right), available - n_left)
        n_left = min(len(left), available - n_right)

        left_part = left[-n_left:] if n_left else []
        right_part = right[:n_right]

        ids = (
            [BOS_ID]
            + previous_prefix + left_part + [EOS_ID]
            + next_prefix + right_part + [EOS_ID]
            + target_prefix + part + [EOS_ID]
        )

        chunks.append((ids, len(part)))

    return chunks


class RRDataset(Dataset):
    def __init__(self, documents):
        self.documents = documents
        self.tokens = []
        self.index = []

        for doc_index, document in enumerate(documents):
            encoded = []

            for sentence_index, sentence in enumerate(document["text"]):
                ids = tokenizer.encode(sentence)
                encoded.append(ids or [UNK_ID])

                self.index.append(
                    (doc_index, sentence_index)
                )

            self.tokens.append(encoded)

    def __len__(self):
        return len(self.index)

    def __getitem__(self, index):
        doc_index, sentence_index = self.index[index]

        document = self.documents[doc_index]
        sentences = self.tokens[doc_index]

        left = []
        right = []

        if USE_CONTEXT:
            if sentence_index > 0:
                left = sentences[sentence_index - 1]

            if sentence_index + 1 < len(sentences):
                right = sentences[sentence_index + 1]

        chunks = make_chunks(
            sentences[sentence_index], left, right
        )

        label = -100
        if document["labels"] is not None:
            label = label2id[
                document["labels"][sentence_index]
            ]

        return {
            "chunks": chunks,
            "label": label,
            "meta": (
                document["domain"],
                document["id"],
                sentence_index,
            ),
        }


def collate_rr(examples):
    input_ids = []
    lengths = []
    owners = []
    weights = []

    for owner, example in enumerate(examples):
        for ids, target_length in example["chunks"]:
            input_ids.append(torch.tensor(ids, dtype=torch.long))
            lengths.append(len(ids))
            owners.append(owner)
            weights.append(target_length)

    return {
        "input_ids": pad_sequence(
            input_ids,
            batch_first=True,
            padding_value=PAD_ID,
        ),
        "lengths": torch.tensor(lengths),
        "owners": torch.tensor(owners),
        "weights": torch.tensor(weights, dtype=torch.float32),
        "labels": torch.tensor(
            [example["label"] for example in examples]
        ),
        "meta": [example["meta"] for example in examples],
    }


train_dataset = RRDataset(train_docs)
dev_dataset = RRDataset(dev_docs)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_rr,
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_rr,
)

print("Training sentences:", len(train_dataset))
print("Development sentences:", len(dev_dataset))

Training sentences: 16845
Development sentences: 2142


In [11]:
def rr_attention_forward(self, x):
    batch_size, length, hidden_size = x.shape

    q = self.q_proj(x)
    k = self.k_proj(x)
    v = self.v_proj(x)

    q = q.view(
        batch_size, length, self.n_heads, self.d_head
    ).transpose(1, 2)

    k = k.view(
        batch_size, length, self.n_heads, self.d_head
    ).transpose(1, 2)

    v = v.view(
        batch_size, length, self.n_heads, self.d_head
    ).transpose(1, 2)

    q, k = self.apply_rope(q, k)

    output = F.scaled_dot_product_attention(
        q, k, v,
        is_causal=True,
        dropout_p=self.attn_dropout.p if self.training else 0.0,
        scale=self.attention_scale,
    )

    output = output.transpose(1, 2).contiguous()
    output = output.view(batch_size, length, hidden_size)

    return self.resid_dropout(self.out_proj(output))


for block in phase1_model.blocks:
    block.attn.forward = MethodType(
        rr_attention_forward, block.attn
    )

In [12]:
class LoRA(nn.Module):
    def __init__(self, base_layer, rank, alpha):
        super().__init__()

        self.base = base_layer
        self.base.requires_grad_(False)
        self.scale = alpha / rank

        self.A = nn.Linear(
            base_layer.in_features, rank, bias=False
        )
        self.B = nn.Linear(
            rank, base_layer.out_features, bias=False
        )

        nn.init.normal_(self.A.weight, std=0.02)
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        return self.base(x) + self.scale * self.B(self.A(x))

class LoRMAPlus(nn.Module):
    def __init__(self, base_layer, rank, alpha):
        super().__init__()

        self.base = base_layer
        self.base.requires_grad_(False)

        self.scale = alpha / rank
        out_features = base_layer.out_features

        self.A = nn.Linear(
            out_features,
            rank,
            bias=False,
        )

        self.B = nn.Linear(
            rank,
            out_features,
            bias=False,
        )

        self.A.to(
            device=base_layer.weight.device,
            dtype=base_layer.weight.dtype,
        )
        self.B.to(
            device=base_layer.weight.device,
            dtype=base_layer.weight.dtype,
        )

        nn.init.normal_(self.A.weight, std=0.02)
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        original = self.base(x)
        correction = self.B(self.A(original))

        return original + self.scale * correction


class Model1RR(nn.Module):
    def __init__(self, pretrained_model, num_labels):
        super().__init__()

        self.cfg = pretrained_model.cfg
        self.tok_emb = pretrained_model.tok_emb
        self.blocks = pretrained_model.blocks
        self.ln_f = pretrained_model.ln_f

        self.dropout = nn.Dropout(self.cfg.dropout)
        self.classifier = nn.Linear(
            self.cfg.d_model, num_labels
        )

    def encode_sentences(self, input_ids, lengths, target_lengths):
        x = self.tok_emb(input_ids)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x).float()

        # Target tokens occur immediately before the final EOS
        target_ends = lengths - 1
        target_starts = target_ends - target_lengths

        positions = torch.arange(
            input_ids.size(1),
            device=input_ids.device,
        )

        target_mask = (
            (positions[None, :] >= target_starts[:, None])
            & (positions[None, :] < target_ends[:, None])
        )

        mask = target_mask.unsqueeze(-1).float()

        target_mean = (
            (x * mask).sum(dim=1)
            / mask.sum(dim=1).clamp_min(1)
        )

        return target_mean

    def forward(self, input_ids, lengths, target_lengths):
        features = self.encode_sentences(
            input_ids,
            lengths,
            target_lengths,
        )

        return self.classifier(self.dropout(features))


adapter_classes = {
    "lora": LoRA,
    "lorma_plus": LoRMAPlus,
}

Adapter = adapter_classes[METHOD]

torch.manual_seed(SEED)

rr_model = Model1RR(
    phase1_model,
    NUM_LABELS,
)

del phase1_model

# Freeze pretrained parameters
rr_model.requires_grad_(False)

ATTENTION_TARGETS = [
    "q_proj",
    "k_proj",
    "v_proj",
    "out_proj",
]

FFN_TARGETS = [0, 2]

# Add fresh trainable adapters
for block in rr_model.blocks:
    for name in ATTENTION_TARGETS:
        layer = getattr(block.attn, name)

        setattr(
            block.attn,
            name,
            Adapter(
                layer,
                rank=RANK,
                alpha=ALPHA,
            ),
        )

    for index in FFN_TARGETS:
        block.ffn.net[index] = Adapter(
            block.ffn.net[index],
            rank=RANK,
            alpha=ALPHA,
        )

# Train the sentence classifier too
rr_model.classifier.requires_grad_(True)
rr_model = rr_model.to(device)

total = sum(
    p.numel()
    for p in rr_model.parameters()
)

trainable = sum(
    p.numel()
    for p in rr_model.parameters()
    if p.requires_grad
)

print("Method:", METHOD)
print(f"Total parameters: {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Trainable percentage: {100 * trainable / total:.2f}%")

Method: lora
Total parameters: 57,394,973
Trainable parameters: 974,973
Trainable percentage: 1.70%


In [13]:
def move_batch(batch):
    return {
        key: value.to(device) if torch.is_tensor(value) else value
        for key, value in batch.items()
    }


def get_sentence_logits(model, batch):
    # chunk_logits = model(
    #     batch["input_ids"], batch["lengths"]
    # ).float()

    chunk_logits = model(
        batch["input_ids"],
        batch["lengths"],
        batch["weights"].long(),
    ).float()

    owners = batch["owners"]
    weights = batch["weights"]
    num_sentences = len(batch["meta"])

    logits = chunk_logits.new_zeros(
        num_sentences, NUM_LABELS
    )

    logits = logits.index_add(
        0,
        owners,
        chunk_logits * weights[:, None],
    )

    totals = weights.new_zeros(num_sentences)
    totals = totals.index_add(0, owners, weights)

    return logits / totals[:, None]


counts = torch.zeros(NUM_LABELS)

for document in train_docs:
    for label in document["labels"]:
        counts[label2id[label]] += 1

class_weights = None

if USE_CLASS_WEIGHTS:
    class_weights = counts.clamp_min(1).rsqrt()
    class_weights /= class_weights.mean()
    class_weights = class_weights.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

print(dict(zip(label_names, counts.int().tolist())))

{'Fact': 4098, 'Issue': 107, 'ArgumentPetitioner': 1613, 'ArgumentRespondent': 1348, 'Statute': 1099, 'Dissent': 284, 'PrecedentReliedUpon': 1945, 'PrecedentNotReliedUpon': 569, 'PrecedentOverruled': 3, 'RulingByLowerCourt': 874, 'RatioOfTheDecision': 4301, 'RulingByPresentCourt': 307, 'None': 297}


In [14]:
def macro_scores(gold, predicted):
    precision, recall, f1, _ = precision_recall_fscore_support(
        gold,
        predicted,
        labels=list(range(NUM_LABELS)),
        average="macro",
        zero_division=0,
    )

    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }


@torch.inference_mode()
def evaluate_rr(model, loader):
    model.eval()

    gold = []
    predicted = []
    domains = []

    for batch in loader:
        batch = move_batch(batch)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            logits = get_sentence_logits(model, batch)

        gold.extend(batch["labels"].tolist())
        predicted.extend(logits.argmax(-1).tolist())
        domains.extend(meta[0] for meta in batch["meta"])

    scores = macro_scores(gold, predicted)
    scores["domains"] = {}

    for domain in ("CL", "IT"):
        indices = [
            i for i, name in enumerate(domains)
            if name == domain
        ]

        if indices:
            scores["domains"][domain] = macro_scores(
                [gold[i] for i in indices],
                [predicted[i] for i in indices],
            )

    return scores, gold, predicted

In [15]:
trainable_parameters = [
    p for p in rr_model.parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LEARNING_RATE,
    weight_decay=0.01,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp and amp_dtype == torch.float16,
)

run_config = {
    "method": METHOD,
    "adapter_class": Adapter.__name__,
    "rank": RANK,
    "alpha": ALPHA,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "use_context": USE_CONTEXT,
    "use_class_weights": USE_CLASS_WEIGHTS,
    "seed": SEED,
    # "pooling": "final_eos",
    "pooling": "target_mean",
    "input_order": "previous_next_target",
}

(OUTPUT_DIR / "run_config.json").write_text(
    json.dumps(run_config, indent=2),
    encoding="utf-8",
)

history = []
best_f1 = -1.0
bad_epochs = 0

for epoch in range(1, EPOCHS + 1):
    rr_model.train()

    total_loss = 0.0
    total_weight = 0.0

    for step, batch in enumerate(train_loader, 1):
        batch = move_batch(batch)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            logits = get_sentence_logits(rr_model, batch)
            loss = criterion(logits, batch["labels"])

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            trainable_parameters, 1.0
        )

        scaler.step(optimizer)
        scaler.update()

        weight = (
            len(batch["meta"])
            if class_weights is None
            else class_weights[batch["labels"]].sum().item()
        )

        total_loss += loss.item() * weight
        total_weight += weight

        if step % 500 == 0:
            print(
                f"Epoch {epoch} | "
                f"Batch {step}/{len(train_loader)} | "
                f"Loss: {loss.item():.4f}"
            )

    scores, _, _ = evaluate_rr(rr_model, dev_loader)
    train_loss = total_loss / total_weight

    print(
        f"Epoch {epoch} | "
        f"Train loss: {train_loss:.4f} | "
        f"Dev Macro-F1: {scores['f1']:.4f}"
    )
    print("Domain scores:", scores["domains"])

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        **scores,
    })

    (OUTPUT_DIR / "history.json").write_text(
        json.dumps(history, indent=2),
        encoding="utf-8",
    )

    if scores["f1"] > best_f1:
        best_f1 = scores["f1"]
        bad_epochs = 0

        torch.save(
            {
                "model_state_dict": {
                    name: tensor.detach().cpu()
                    for name, tensor in rr_model.state_dict().items()
                },
                "model_config": asdict(cfg),
                "run_config": run_config,
                "label2id": label2id,
                "epoch": epoch,
                "dev_scores": scores,
            },
            OUTPUT_DIR / "best_f1.pt",
        )

        print("Saved best checkpoint.")

    else:
        bad_epochs += 1

        if bad_epochs >= PATIENCE:
            print("Early stopping.")
            break

Epoch 1 | Batch 500/1053 | Loss: 1.4408
Epoch 1 | Batch 1000/1053 | Loss: 1.4491
Epoch 1 | Train loss: 1.5563 | Dev Macro-F1: 0.3095
Domain scores: {'CL': {'precision': 0.26407156184664937, 'recall': 0.2683996454441, 'f1': 0.23861063486404963}, 'IT': {'precision': 0.4023780451832665, 'recall': 0.3226916506911651, 'f1': 0.3305337985968911}}
Saved best checkpoint.
Epoch 2 | Batch 500/1053 | Loss: 1.2302
Epoch 2 | Batch 1000/1053 | Loss: 1.4110
Epoch 2 | Train loss: 1.1200 | Dev Macro-F1: 0.3677
Domain scores: {'CL': {'precision': 0.28208682399482415, 'recall': 0.2846543696007782, 'f1': 0.2662125402592202}, 'IT': {'precision': 0.49774398379763046, 'recall': 0.38929608271878197, 'f1': 0.4137394614123367}}
Saved best checkpoint.
Epoch 3 | Batch 500/1053 | Loss: 0.6005
Epoch 3 | Batch 1000/1053 | Loss: 0.7463
Epoch 3 | Train loss: 0.9346 | Dev Macro-F1: 0.3821
Domain scores: {'CL': {'precision': 0.278624964533705, 'recall': 0.29680772930399446, 'f1': 0.2719018169149675}, 'IT': {'precision': 

In [16]:
checkpoint = torch.load(
    OUTPUT_DIR / "best_f1.pt",
    map_location="cpu",
    weights_only=True,
)

rr_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

print("Best epoch:", checkpoint["epoch"])
del checkpoint

scores, gold, predicted = evaluate_rr(
    rr_model, dev_loader
)

print(
    classification_report(
        gold,
        predicted,
        labels=list(range(NUM_LABELS)),
        target_names=label_names,
        digits=4,
        zero_division=0,
    )
)

print("Domain scores:", scores["domains"])

Best epoch: 3
                        precision    recall  f1-score   support

                  Fact     0.6082    0.6329    0.6203       444
                 Issue     0.6667    0.5333    0.5926        15
    ArgumentPetitioner     0.6019    0.3175    0.4157       400
    ArgumentRespondent     0.4724    0.3456    0.3992       272
               Statute     0.7705    0.7833    0.7769       120
               Dissent     0.0000    0.0000    0.0000         0
   PrecedentReliedUpon     0.4545    0.4134    0.4330       254
PrecedentNotReliedUpon     0.3158    0.2400    0.2727        50
    PrecedentOverruled     0.0000    0.0000    0.0000         0
    RulingByLowerCourt     0.3462    0.1765    0.2338       102
    RatioOfTheDecision     0.3932    0.7171    0.5079       403
  RulingByPresentCourt     0.5000    0.5306    0.5149        49
                  None     0.2222    0.1818    0.2000        33

              accuracy                         0.4949      2142
             macro avg  

In [17]:
test_docs = []

for domain in ("CL", "IT"):
    test_docs += read_documents(
        RR_DIR / f"Test/{domain}_test.json",
        domain,
        labeled=False,
    )

test_dataset = RRDataset(test_docs)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_rr,
)


@torch.inference_mode()
def predict_rr(model, loader, documents):
    model.eval()

    predictions = {
        (doc["domain"], doc["id"]): [None] * len(doc["text"])
        for doc in documents
    }

    for batch in loader:
        batch = move_batch(batch)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            logits = get_sentence_logits(model, batch)

        predicted_ids = logits.argmax(-1).tolist()

        for meta, label_id in zip(batch["meta"], predicted_ids):
            domain, doc_id, sentence_index = meta
            predictions[(domain, doc_id)][sentence_index] = (
                id2label[label_id]
            )

    return predictions


predictions = predict_rr(
    rr_model, test_loader, test_docs
)

submission = {}

for (domain, doc_id), labels in predictions.items():
    if doc_id in submission:
        raise ValueError(
            f"Document ID {doc_id} occurs in both domains; "
            "check the official submission format."
        )

    if any(label is None for label in labels):
        raise ValueError(f"Missing predictions for {doc_id}")

    submission[doc_id] = labels

prediction_path = OUTPUT_DIR / "prediction.json"

prediction_path.write_text(
    json.dumps(submission, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

with zipfile.ZipFile(
    OUTPUT_DIR / "submission.zip",
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(
        prediction_path,
        arcname="prediction.json",
    )

print("Saved:", OUTPUT_DIR / "submission.zip")

Saved: /kaggle/working/phase3_model1_lora_target_mean/submission.zip


# **BI GRU**

In [18]:
M_OUTPUT_DIR = Path("/kaggle/input/datasets/bokkatrishalreddy/lora-saved-files")

In [19]:
SENTENCE_CHECKPOINT = OUTPUT_DIR / "best_f1.pt"

checkpoint = torch.load(
    SENTENCE_CHECKPOINT,
    map_location="cpu",
    weights_only=True,
)

rr_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

print("Best sentence-model epoch:", checkpoint["epoch"])
print("Sentence-model scores:", checkpoint["dev_scores"])

del checkpoint

rr_model = rr_model.to(device)
rr_model.requires_grad_(False)
rr_model.eval()

BIGRU_DIR = OUTPUT_DIR / "document_bigru"
BIGRU_DIR.mkdir(parents=True, exist_ok=True)

DOC_BATCH_SIZE = 4
GRU_HIDDEN = 512
GRU_DROPOUT = 0.2
GRU_LR = 1e-3
GRU_EPOCHS = 15
GRU_PATIENCE = 3

Best sentence-model epoch: 3
Sentence-model scores: {'precision': 0.41165764594847265, 'recall': 0.37477278574277584, 'f1': 0.38206670247592683, 'domains': {'CL': {'precision': 0.278624964533705, 'recall': 0.29680772930399446, 'f1': 0.2719018169149675}, 'IT': {'precision': 0.4681596041793691, 'recall': 0.4088101225203393, 'f1': 0.42527128696285627}}}


In [20]:
@torch.no_grad()
def extract_document_features(model, documents):
    model.eval()

    loader = DataLoader(
        RRDataset(documents),
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_rr,
    )

    document_features = {
        (doc["domain"], doc["id"]): torch.zeros(
            len(doc["text"]),
            model.cfg.d_model,
        )
        for doc in documents
    }

    for step, batch in enumerate(loader, 1):
        batch = move_batch(batch)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            chunk_features = model.encode_sentences(
                batch["input_ids"],
                batch["lengths"],
                batch["weights"].long(),
            ).float()

        owners = batch["owners"]
        weights = batch["weights"]

        features = chunk_features.new_zeros(
            len(batch["meta"]),
            model.cfg.d_model,
        )

        features.index_add_(
            0,
            owners,
            chunk_features * weights[:, None],
        )

        totals = weights.new_zeros(len(batch["meta"]))
        totals.index_add_(0, owners, weights)

        features = (features / totals[:, None]).cpu()

        for meta, vector in zip(batch["meta"], features):
            domain, doc_id, sentence_index = meta

            document_features[
                (domain, doc_id)
            ][sentence_index] = vector

        if step % 100 == 0:
            print(f"Extracted {step}/{len(loader)} batches")

    records = []

    for doc in documents:
        if not doc["text"]:
            continue

        key = (doc["domain"], doc["id"])

        labels = (
            [label2id[label] for label in doc["labels"]]
            if doc["labels"] is not None
            else [-100] * len(doc["text"])
        )

        records.append({
            "features": document_features[key],
            "labels": torch.tensor(labels, dtype=torch.long),
            "key": key,
        })

    return records


train_records = extract_document_features(
    rr_model, train_docs
)

dev_records = extract_document_features(
    rr_model, dev_docs
)

Extracted 100/1053 batches
Extracted 200/1053 batches
Extracted 300/1053 batches
Extracted 400/1053 batches
Extracted 500/1053 batches
Extracted 600/1053 batches
Extracted 700/1053 batches
Extracted 800/1053 batches
Extracted 900/1053 batches
Extracted 1000/1053 batches
Extracted 100/134 batches


In [21]:
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence,
)


def collate_documents(records):
    return {
        "features": pad_sequence(
            [record["features"] for record in records],
            batch_first=True,
            padding_value=0.0,
        ),
        "labels": pad_sequence(
            [record["labels"] for record in records],
            batch_first=True,
            padding_value=-100,
        ),
        "lengths": torch.tensor(
            [len(record["labels"]) for record in records],
            dtype=torch.long,
        ),
        "keys": [record["key"] for record in records],
    }


doc_train_loader = DataLoader(
    train_records,
    batch_size=DOC_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_documents,
)

doc_dev_loader = DataLoader(
    dev_records,
    batch_size=DOC_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_documents,
)


class DocumentBiGRU(nn.Module):
    def __init__(
        self,
        input_size,
        hidden_size,
        num_labels,
        dropout=0.2,
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            2 * hidden_size,
            num_labels,
        )

    def forward(self, features, lengths):
        packed = pack_padded_sequence(
            features,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        packed_output, _ = self.gru(packed)

        output, _ = pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=features.size(1),
        )

        return self.classifier(self.dropout(output))


torch.manual_seed(SEED)

doc_model = DocumentBiGRU(
    input_size=rr_model.cfg.d_model,
    hidden_size=GRU_HIDDEN,
    num_labels=NUM_LABELS,
    dropout=GRU_DROPOUT,
).to(device)

In [22]:
@torch.no_grad()
def evaluate_bigru(model, loader):
    model.eval()

    gold = []
    predicted = []

    for batch in loader:
        features = batch["features"].to(device)
        labels = batch["labels"].to(device)

        logits = model(features, batch["lengths"])
        predictions = logits.argmax(dim=-1)

        valid = labels.ne(-100)

        gold.extend(labels[valid].cpu().tolist())
        predicted.extend(predictions[valid].cpu().tolist())

    return macro_scores(gold, predicted), gold, predicted


doc_criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    ignore_index=-100,
)

doc_optimizer = torch.optim.AdamW(
    doc_model.parameters(),
    lr=GRU_LR,
    weight_decay=0.01,
)

best_doc_f1 = -1.0
bad_epochs = 0

for epoch in range(1, GRU_EPOCHS + 1):
    doc_model.train()

    total_loss = 0.0
    total_weight = 0.0

    for batch in doc_train_loader:
        features = batch["features"].to(device)
        labels = batch["labels"].to(device)

        doc_optimizer.zero_grad(set_to_none=True)

        logits = doc_model(features, batch["lengths"])

        loss = doc_criterion(
            logits.reshape(-1, NUM_LABELS),
            labels.reshape(-1),
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            doc_model.parameters(), 1.0
        )

        doc_optimizer.step()

        valid_labels = labels[labels.ne(-100)]

        weight = (
            valid_labels.numel()
            if class_weights is None
            else class_weights[valid_labels].sum().item()
        )

        total_loss += loss.item() * weight
        total_weight += weight

    scores, _, _ = evaluate_bigru(
        doc_model, doc_dev_loader
    )

    print(
        f"Epoch {epoch} | "
        f"Train loss: {total_loss / total_weight:.4f} | "
        f"Dev Macro-F1: {scores['f1']:.4f}"
    )

    if scores["f1"] > best_doc_f1:
        best_doc_f1 = scores["f1"]
        bad_epochs = 0

        torch.save(
            {
                "model_state_dict": {
                    name: tensor.detach().cpu()
                    for name, tensor in doc_model.state_dict().items()
                },
                "input_size": rr_model.cfg.d_model,
                "hidden_size": GRU_HIDDEN,
                "dropout": GRU_DROPOUT,
                "num_labels": NUM_LABELS,
                "label2id": label2id,
                "pooling": "target_mean",
                "sentence_checkpoint": str(SENTENCE_CHECKPOINT),
                "epoch": epoch,
                "dev_scores": scores,
            },
            BIGRU_DIR / "best_bigru.pt",
        )

        print("Saved best BiGRU checkpoint.")

    else:
        bad_epochs += 1

        if bad_epochs >= GRU_PATIENCE:
            print("Early stopping.")
            break

Epoch 1 | Train loss: 1.2814 | Dev Macro-F1: 0.3878
Saved best BiGRU checkpoint.
Epoch 2 | Train loss: 0.7194 | Dev Macro-F1: 0.3853
Epoch 3 | Train loss: 0.5816 | Dev Macro-F1: 0.4054
Saved best BiGRU checkpoint.
Epoch 4 | Train loss: 0.5033 | Dev Macro-F1: 0.4390
Saved best BiGRU checkpoint.
Epoch 5 | Train loss: 0.4276 | Dev Macro-F1: 0.4296
Epoch 6 | Train loss: 0.3519 | Dev Macro-F1: 0.4347
Epoch 7 | Train loss: 0.2916 | Dev Macro-F1: 0.4570
Saved best BiGRU checkpoint.
Epoch 8 | Train loss: 0.2620 | Dev Macro-F1: 0.4287
Epoch 9 | Train loss: 0.2223 | Dev Macro-F1: 0.4163
Epoch 10 | Train loss: 0.1928 | Dev Macro-F1: 0.4632
Saved best BiGRU checkpoint.
Epoch 11 | Train loss: 0.1817 | Dev Macro-F1: 0.4379
Epoch 12 | Train loss: 0.1690 | Dev Macro-F1: 0.4419
Epoch 13 | Train loss: 0.1540 | Dev Macro-F1: 0.4589
Early stopping.


In [23]:
checkpoint = torch.load(
    BIGRU_DIR / "best_bigru.pt",
    map_location="cpu",
    weights_only=True,
)

doc_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

print("Best BiGRU epoch:", checkpoint["epoch"])
del checkpoint

scores, gold, predicted = evaluate_bigru(
    doc_model, doc_dev_loader
)

print("BiGRU validation macro F1:", scores["f1"])

print(
    classification_report(
        gold,
        predicted,
        labels=list(range(NUM_LABELS)),
        target_names=label_names,
        digits=4,
        zero_division=0,
    )
)

Best BiGRU epoch: 10
BiGRU validation macro F1: 0.4632116091438578
                        precision    recall  f1-score   support

                  Fact     0.6848    0.6802    0.6825       444
                 Issue     0.7000    0.4667    0.5600        15
    ArgumentPetitioner     0.7565    0.5125    0.6110       400
    ArgumentRespondent     0.6769    0.5699    0.6188       272
               Statute     0.7121    0.7833    0.7460       120
               Dissent     0.0000    0.0000    0.0000         0
   PrecedentReliedUpon     0.5941    0.3976    0.4764       254
PrecedentNotReliedUpon     0.3279    0.4000    0.3604        50
    PrecedentOverruled     0.0000    0.0000    0.0000         0
    RulingByLowerCourt     0.6613    0.4020    0.5000       102
    RatioOfTheDecision     0.4993    0.8486    0.6287       403
  RulingByPresentCourt     0.5918    0.5918    0.5918        49
                  None     0.2500    0.2424    0.2462        33

              accuracy             

In [24]:
test_docs = []

for domain in ("CL", "IT"):
    test_docs += read_documents(
        RR_DIR / f"Test/{domain}_test.json",
        domain,
        labeled=False,
    )

test_records = extract_document_features(
    rr_model, test_docs
)

doc_test_loader = DataLoader(
    test_records,
    batch_size=DOC_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_documents,
)


@torch.no_grad()
def predict_bigru(model, loader, documents):
    model.eval()

    predictions = {
        (doc["domain"], doc["id"]): []
        for doc in documents
    }

    for batch in loader:
        features = batch["features"].to(device)

        logits = model(features, batch["lengths"])
        predicted_ids = logits.argmax(dim=-1).cpu()

        for row, (key, length) in enumerate(
            zip(batch["keys"], batch["lengths"].tolist())
        ):
            predictions[key] = [
                id2label[label_id]
                for label_id in predicted_ids[row, :length].tolist()
            ]

    return predictions


predictions = predict_bigru(
    doc_model,
    doc_test_loader,
    test_docs,
)

submission = {}

for (domain, doc_id), labels in predictions.items():
    if doc_id in submission:
        raise ValueError(f"Duplicate document ID: {doc_id}")

    submission[doc_id] = labels

prediction_path = BIGRU_DIR / "prediction.json"

prediction_path.write_text(
    json.dumps(submission, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

with zipfile.ZipFile(
    BIGRU_DIR / "submission.zip",
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(
        prediction_path,
        arcname="prediction.json",
    )

print("Saved:", BIGRU_DIR / "submission.zip")

Extracted 100/138 batches
Saved: /kaggle/working/phase3_model1_lora_target_mean/document_bigru/submission.zip


# **Knowledge Distillation**

In [41]:
import torch.nn.functional as F

STUDENT_HIDDEN = 512
STUDENT_DROPOUT = 0.2

KD_TEMPERATURE = 2.0
KD_WEIGHT = 0.5

KD_LR = 1e-3
KD_EPOCHS = 20
KD_PATIENCE = 4

KD_DIR = BIGRU_DIR / (
    f"kd_hidden{STUDENT_HIDDEN}_weight{KD_WEIGHT:g}"
)

KD_DIR.mkdir(parents=True, exist_ok=True)

In [42]:
teacher_checkpoint = torch.load(
    BIGRU_DIR / "best_bigru.pt",
    map_location="cpu",
    weights_only=True,
)

if teacher_checkpoint["label2id"] != label2id:
    raise ValueError("Teacher label mapping does not match.")

doc_model.load_state_dict(
    teacher_checkpoint["model_state_dict"],
    strict=True,
)

teacher_model = doc_model.to(device)
teacher_model.requires_grad_(False)
teacher_model.eval()

teacher_input_size = teacher_model.gru.input_size
teacher_hidden_size = teacher_model.gru.hidden_size

teacher_sentence_checkpoint = teacher_checkpoint[
    "sentence_checkpoint"
]

print("Teacher hidden size:", teacher_hidden_size)
print("Teacher best epoch:", teacher_checkpoint["epoch"])
print("Saved teacher scores:", teacher_checkpoint["dev_scores"])

del teacher_checkpoint

teacher_scores, _, _ = evaluate_bigru(
    teacher_model,
    doc_dev_loader,
)

print("Teacher validation macro F1:", teacher_scores["f1"])

Teacher hidden size: 512
Teacher best epoch: 10
Saved teacher scores: {'precision': 0.49650965830109656, 'recall': 0.4534636856766227, 'f1': 0.4632116091438578}
Teacher validation macro F1: 0.4632116091438578


In [43]:
@torch.no_grad()
def cache_teacher_logits(model, loader):
    model.eval()
    cached_logits = {}

    for batch in loader:
        features = batch["features"].to(device)

        logits = model(
            features,
            batch["lengths"],
        ).float().cpu()

        for row, (key, length) in enumerate(
            zip(batch["keys"], batch["lengths"].tolist())
        ):
            cached_logits[key] = logits[row, :length].clone()

    return cached_logits


teacher_train_logits = cache_teacher_logits(
    teacher_model,
    doc_train_loader,
)

print("Cached training documents:", len(teacher_train_logits))

# The teacher is no longer needed on the GPU during training
teacher_model.cpu()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Cached training documents: 80


In [44]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

student_model = DocumentBiGRU(
    input_size=teacher_input_size,
    hidden_size=STUDENT_HIDDEN,
    num_labels=NUM_LABELS,
    dropout=STUDENT_DROPOUT,
).to(device)

teacher_parameters = sum(
    p.numel() for p in teacher_model.parameters()
)

student_parameters = sum(
    p.numel() for p in student_model.parameters()
)

print(f"Teacher BiGRU parameters: {teacher_parameters:,}")
print(f"Student BiGRU parameters: {student_parameters:,}")

print(
    "BiGRU parameter reduction:",
    f"{100 * (1 - student_parameters / teacher_parameters):.2f}%",
)

Teacher BiGRU parameters: 3,312,653
Student BiGRU parameters: 3,312,653
BiGRU parameter reduction: 0.00%


In [45]:
student_criterion = nn.CrossEntropyLoss(
    weight=class_weights,
)


def distillation_loss(student_logits, teacher_logits, labels):
    # Exclude padded sentences from both losses
    valid = labels.ne(-100)

    student_valid = student_logits[valid].float()
    teacher_valid = teacher_logits[valid].float()
    gold_valid = labels[valid]

    ce_loss = student_criterion(
        student_valid,
        gold_valid,
    )

    kd_loss = F.kl_div(
        F.log_softmax(
            student_valid / KD_TEMPERATURE,
            dim=-1,
        ),
        F.softmax(
            teacher_valid / KD_TEMPERATURE,
            dim=-1,
        ),
        reduction="batchmean",
    ) * (KD_TEMPERATURE ** 2)

    loss = (
        (1 - KD_WEIGHT) * ce_loss
        + KD_WEIGHT * kd_loss
    )

    return loss

In [46]:
student_optimizer = torch.optim.AdamW(
    student_model.parameters(),
    lr=KD_LR,
    weight_decay=0.01,
)

kd_config = {
    "student_hidden_size": STUDENT_HIDDEN,
    "student_dropout": STUDENT_DROPOUT,
    "teacher_hidden_size": teacher_hidden_size,
    "temperature": KD_TEMPERATURE,
    "kd_weight": KD_WEIGHT,
    "learning_rate": KD_LR,
    "seed": SEED,
    "pooling": "target_mean",
    "teacher_checkpoint": str(BIGRU_DIR / "best_bigru.pt"),
    "sentence_checkpoint": teacher_sentence_checkpoint,
    "use_class_weights": class_weights is not None,
}

(KD_DIR / "run_config.json").write_text(
    json.dumps(kd_config, indent=2),
    encoding="utf-8",
)

best_student_f1 = -1.0
bad_epochs = 0
kd_history = []

for epoch in range(1, KD_EPOCHS + 1):
    student_model.train()
    total_batch_loss = 0.0

    for batch in doc_train_loader:
        features = batch["features"].to(device)
        labels = batch["labels"].to(device)

        # Same document order and padding as the current batch
        teacher_logits = pad_sequence(
            [
                teacher_train_logits[key]
                for key in batch["keys"]
            ],
            batch_first=True,
            padding_value=0.0,
        ).to(device)

        student_optimizer.zero_grad(set_to_none=True)

        student_logits = student_model(
            features,
            batch["lengths"],
        )

        loss = distillation_loss(
            student_logits,
            teacher_logits,
            labels,
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            student_model.parameters(),
            1.0,
        )

        student_optimizer.step()
        total_batch_loss += loss.item()

    scores, _, _ = evaluate_bigru(
        student_model,
        doc_dev_loader,
    )

    mean_batch_loss = total_batch_loss / len(doc_train_loader)

    print(
        f"Epoch {epoch} | "
        f"Mean batch loss: {mean_batch_loss:.4f} | "
        f"Dev Macro-F1: {scores['f1']:.4f}"
    )

    kd_history.append({
        "epoch": epoch,
        "mean_batch_loss": mean_batch_loss,
        **scores,
    })

    (KD_DIR / "history.json").write_text(
        json.dumps(kd_history, indent=2),
        encoding="utf-8",
    )

    if scores["f1"] > best_student_f1:
        best_student_f1 = scores["f1"]
        bad_epochs = 0

        torch.save(
            {
                "model_state_dict": {
                    name: tensor.detach().cpu()
                    for name, tensor in student_model.state_dict().items()
                },
                "input_size": teacher_input_size,
                "hidden_size": STUDENT_HIDDEN,
                "dropout": STUDENT_DROPOUT,
                "num_labels": NUM_LABELS,
                "label2id": label2id,
                "pooling": "target_mean",
                "sentence_checkpoint": teacher_sentence_checkpoint,
                "run_config": kd_config,
                "epoch": epoch,
                "dev_scores": scores,
            },
            KD_DIR / "best_student.pt",
        )

        print("Saved best student checkpoint.")

    else:
        bad_epochs += 1

        if bad_epochs >= KD_PATIENCE:
            print("Early stopping.")
            break

Epoch 1 | Mean batch loss: 1.8195 | Dev Macro-F1: 0.3736
Saved best student checkpoint.
Epoch 2 | Mean batch loss: 0.9138 | Dev Macro-F1: 0.4006
Saved best student checkpoint.
Epoch 3 | Mean batch loss: 0.6741 | Dev Macro-F1: 0.4138
Saved best student checkpoint.
Epoch 4 | Mean batch loss: 0.6198 | Dev Macro-F1: 0.4258
Saved best student checkpoint.
Epoch 5 | Mean batch loss: 0.4512 | Dev Macro-F1: 0.4338
Saved best student checkpoint.
Epoch 6 | Mean batch loss: 0.3363 | Dev Macro-F1: 0.4476
Saved best student checkpoint.
Epoch 7 | Mean batch loss: 0.2778 | Dev Macro-F1: 0.4566
Saved best student checkpoint.
Epoch 8 | Mean batch loss: 0.2557 | Dev Macro-F1: 0.4429
Epoch 9 | Mean batch loss: 0.2658 | Dev Macro-F1: 0.4524
Epoch 10 | Mean batch loss: 0.2267 | Dev Macro-F1: 0.4662
Saved best student checkpoint.
Epoch 11 | Mean batch loss: 0.1947 | Dev Macro-F1: 0.4417
Epoch 12 | Mean batch loss: 0.1887 | Dev Macro-F1: 0.4641
Epoch 13 | Mean batch loss: 0.1762 | Dev Macro-F1: 0.4663
Saved b

In [47]:
checkpoint = torch.load(
    KD_DIR / "best_student.pt",
    map_location="cpu",
    weights_only=True,
)

student_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

print("Best student epoch:", checkpoint["epoch"])
del checkpoint

student_scores, gold, predicted = evaluate_bigru(
    student_model,
    doc_dev_loader,
)

print("Teacher macro F1:", teacher_scores["f1"])
print("Student macro F1:", student_scores["f1"])

print(
    classification_report(
        gold,
        predicted,
        labels=list(range(NUM_LABELS)),
        target_names=label_names,
        digits=4,
        zero_division=0,
    )
)

Best student epoch: 13
Teacher macro F1: 0.4632116091438578
Student macro F1: 0.4663418198505308
                        precision    recall  f1-score   support

                  Fact     0.7139    0.6464    0.6785       444
                 Issue     0.6154    0.5333    0.5714        15
    ArgumentPetitioner     0.7607    0.5325    0.6265       400
    ArgumentRespondent     0.6777    0.5257    0.5921       272
               Statute     0.7520    0.7833    0.7673       120
               Dissent     0.0000    0.0000    0.0000         0
   PrecedentReliedUpon     0.5661    0.5394    0.5524       254
PrecedentNotReliedUpon     0.3051    0.3600    0.3303        50
    PrecedentOverruled     0.0000    0.0000    0.0000         0
    RulingByLowerCourt     0.6897    0.3922    0.5000       102
    RatioOfTheDecision     0.5045    0.8437    0.6314       403
  RulingByPresentCourt     0.5745    0.5510    0.5625        49
                  None     0.2581    0.2424    0.2500        33

     

In [48]:
student_predictions = predict_bigru(
    student_model,
    doc_test_loader,
    test_docs,
)

submission = {}

for (domain, doc_id), labels in student_predictions.items():
    if doc_id in submission:
        raise ValueError(f"Duplicate document ID: {doc_id}")

    submission[doc_id] = labels

prediction_path = KD_DIR / "prediction.json"

prediction_path.write_text(
    json.dumps(submission, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

with zipfile.ZipFile(
    KD_DIR / "submission.zip",
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(
        prediction_path,
        arcname="prediction.json",
    )

print("Saved:", KD_DIR / "submission.zip")

Saved: /kaggle/working/phase3_model1_lora_target_mean/document_bigru/kd_hidden512_weight0.5/submission.zip
